# Lotka–Volterra with BayesFlow

## Ecological model

Let $x(t)$ be the prey population and $y(t)$ the predator population. Their dynamics are

$$
\frac{dx}{dt}=\alpha x-\beta xy,
\qquad
\frac{dy}{dt}=-\gamma y+\delta xy.
$$

| Symbol | Definition |
|---|---|
| $x(t)$ | prey population at time $t$ |
| $y(t)$ | predator population at time $t$ |
| $\alpha$ | prey growth rate in the absence of predators |
| $\beta$ | predation rate: loss of prey through encounters |
| $\gamma$ | predator death rate in the absence of prey |
| $\delta$ | predator growth rate from prey encounters |

We infer $\theta=(\alpha,\beta,\gamma,\delta)$. The simulation starts at $(x_0,y_0)=(1,1)$ and runs from $t=0$ to $t=5$. Each parameter has a scaled logit-normal prior on $(0.1,4.0)$. The observation model retains every tenth time point and adds independent Gaussian noise with standard deviation $0.1$.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax"

import keras
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from scipy.integrate import odeint

import bayesflow as bf

SEED = 1234
rng = np.random.default_rng(SEED)

## Simulator functions

In [ ]:
def prior():
    z = rng.normal(size=4)
    theta = 0.1 + 3.9 / (1.0 + np.exp(-z))
    return dict(alpha=theta[0], beta=theta[1], gamma=theta[2], delta=theta[3])


def lotka_volterra_equations(state, t, alpha, beta, gamma, delta):
    x, y = state
    return [alpha * x - beta * x * y, -gamma * y + delta * x * y]


def ecology_model(alpha, beta, gamma, delta, t_span=(0, 5), t_steps=100, initial_state=(1, 1)):
    t = np.linspace(*t_span, t_steps)
    state = odeint(
        lotka_volterra_equations,
        initial_state,
        t,
        args=(alpha, beta, gamma, delta),
    )
    x, y = state.T
    return dict(x=x, y=y, t=t)


def observation_model(x, y, t, subsample=10, noise_scale=0.1):
    observed_indices = np.arange(0, len(t), subsample)
    return dict(
        observed_x=rng.normal(x[observed_indices], noise_scale),
        observed_y=rng.normal(y[observed_indices], noise_scale),
        observed_t=t[observed_indices],
    )

## BayesFlow simulator

In [ ]:
simulator = bf.make_simulator([prior, ecology_model, observation_model])

samples = simulator.sample(1_000)
keras.tree.map_structure(keras.ops.shape, samples)

## Example trajectories

In [ ]:
def trajectory_interval(trajectories, confidence=0.95):
    tail = (1.0 - confidence) / 2.0
    lower, median, upper = np.quantile(trajectories, [tail, 0.5, 1.0 - tail], axis=0)
    return median, lower, upper


def plot_trajectories(samples, confidence=0.95, n_examples=20):
    fig, ax = plt.subplots(figsize=(12, 3))
    sns.despine()
    t = samples["t"][0]

    for key, name, color in [("x", "Prey", "tab:blue"), ("y", "Predator", "darkred")]:
        median, lower, upper = trajectory_interval(samples[key], confidence)
        ax.plot(t, median, color=color, linewidth=2, label=f"Median {name.lower()}")
        ax.fill_between(t, lower, upper, color=color, alpha=0.2, label=f"{confidence:.0%} interval")
        for i in range(min(n_examples, len(samples[key]))):
            ax.plot(t, samples[key][i], color=color, alpha=0.15)

    ax.set(xlabel="Time $t$", ylabel="Population")
    ax.legend(ncol=2)
    return fig, ax


plot_trajectories(samples);

## End-to-end learning of summary statistics

The time-series network learns $s_\phi(x)$ from the raw observed prey, predator, and time series. Flow Matching learns the posterior $p(\theta\mid s_\phi(x))$. Both networks are trained jointly.

In [ ]:
summary_network = bf.networks.TimeSeriesNetwork()
inference_network = bf.networks.FlowMatching()

In [ ]:
adapter = (
    bf.adapters.Adapter()
    .to_array()
    .convert_dtype("float64", "float32")
    .drop(["x", "y", "t"])
    .as_time_series(["observed_x", "observed_y", "observed_t"])
    .concatenate(["alpha", "beta", "gamma", "delta"], into="inference_variables")
    .concatenate(["observed_x", "observed_y", "observed_t"], into="summary_variables")
)

In [ ]:
workflow = bf.BasicWorkflow(
    simulator=simulator,
    adapter=adapter,
    summary_network=summary_network,
    inference_network=inference_network,
)

## Training

In [ ]:
training_data = simulator.sample(8_192)
validation_data = simulator.sample(256)

history = workflow.fit_offline(
    training_data,
    epochs=15,
    batch_size=64,
    validation_data=validation_data,
)
bf.diagnostics.loss(history);

## Posterior diagnostics

In [ ]:
validation_sims = simulator.sample(200)
posterior_draws = workflow.sample(conditions=validation_sims, num_samples=500)
parameter_names = [r"$\alpha$", r"$\beta$", r"$\gamma$", r"$\delta$"]

### Parameter recovery

$$
r_j=\frac{\sum_{i=1}^{N}(\theta_{ij}^{*}-\bar\theta_j^{*})(\hat\theta_{ij}-\bar{\hat\theta}_j)}
{\sqrt{\sum_{i=1}^{N}(\theta_{ij}^{*}-\bar\theta_j^{*})^2}\sqrt{\sum_{i=1}^{N}(\hat\theta_{ij}-\bar{\hat\theta}_j)^2}}.
$$

For dataset $i$ and parameter $j$, $\theta_{ij}^{*}$ is the true value and $\hat\theta_{ij}$ is the inferred posterior median. The plot compares them across $N$ datasets, adds 95% credible intervals, and reports their Pearson correlation $r_j$.

In [ ]:
bf.diagnostics.plots.recovery(
    estimates=posterior_draws,
    targets=validation_sims,
    variable_names=parameter_names,
);

### Calibration

$$
q_{ij}=\frac{1}{S}\sum_{s=1}^{S}\mathbf 1\!\left(\theta_{ij}^{(s)}<\theta_{ij}^{*}\right),
\qquad
\Delta_j(q)=\widehat F_{Q_j}(q)-q
=\frac{1}{N}\sum_{i=1}^{N}\mathbf 1(q_{ij}\le q)-q.
$$

$\theta_{ij}^{*}$ is the true value of parameter $j$ for simulated dataset $i$; $\theta_{ij}^{(s)}$ is posterior draw $s$; $S$ is the number of posterior draws; and $N$ is the number of simulated datasets. Thus, $q_{ij}$ is the posterior rank (quantile) of the truth. For calibrated posteriors, these ranks are uniform, so the plotted ECDF difference $\Delta_j(q)$ should remain near zero. The current distance ranks are equivalent because all parameters are positive.

In [ ]:
bf.diagnostics.plots.calibration_ecdf(
    estimates=posterior_draws,
    targets=validation_sims,
    variable_names=parameter_names,
    difference=True,
    rank_type="distance",
);

### One posterior and its parameter dependencies

$$
p(\theta\mid D_i)\propto p(D_i\mid\theta)\prod_k p(\theta_k),
\qquad \theta=(\alpha,\beta,\gamma,\delta).
$$

The parameters are independent in the prior, but conditioning on the same observations can make them dependent in the posterior. For example, larger $\alpha$ and $\beta$ can offset each other and produce similar prey trajectories. The off-diagonal panels show these dependencies.

In [ ]:
dataset_id = 0
bf.diagnostics.plots.pairs_posterior(
    estimates=posterior_draws,
    targets=validation_sims,
    dataset_id=dataset_id,
    variable_names=parameter_names,
);

### Posterior contraction

$$
C_{ij}=\operatorname{clip}_{[0,1]}\!\left(1-\frac{\operatorname{Var}(\theta_j\mid D_i)}{\operatorname{Var}(\theta_j)}\right).
$$

$C_{ij}$ measures how much the posterior uncertainty for parameter $j$ in dataset $i$ has decreased relative to the prior. Values near one indicate strong contraction; values near zero indicate little information gain.

In [ ]:
bf.diagnostics.plots.z_score_contraction(
    estimates=posterior_draws,
    targets=validation_sims,
    variable_names=parameter_names,
);

### Posterior predictive check

We pass posterior parameter samples through the simulator and check whether the resulting trajectories resemble the observed data $x$.

In [ ]:
def select_dataset(tree, index):
    return {key: value[index] for key, value in tree.items()}


predictive_trajectories = []
for draw_id in range(200):
    theta = {key: posterior_draws[key][dataset_id, draw_id, 0] for key in ["alpha", "beta", "gamma", "delta"]}
    predictive_trajectories.append(ecology_model(**theta))

posterior_predictive = bf.utils.tree_stack(predictive_trajectories, axis=0)
observed = select_dataset(validation_sims, dataset_id)

In [ ]:
fig, ax = plot_trajectories(posterior_predictive)
ax.scatter(observed["observed_t"], observed["observed_x"], color="tab:blue", marker="x", label="Observed prey")
ax.scatter(observed["observed_t"], observed["observed_y"], color="darkred", marker="x", label="Observed predator")
ax.set_title("Posterior predictive check")
ax.legend(ncol=2);